In [6]:
pwd

'C:\\Users\\Thilipan'

In [2]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

In [3]:
with open('input.txt','r',encoding = 'utf-8') as f:
    text = f.read()

In [4]:
len(text)

1115394

In [5]:
device = "cuda" if torch.cuda.is_available else "cpu"

In [6]:
text[:40]

'First Citizen:\nBefore we proceed any fur'

In [7]:
char = sorted(set(text))
vocab_size = len(char)
''.join(char)

"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

In [9]:
stoi = {s:i for i,s in enumerate(char)}
itos = {i:s for s,i in stoi.items()}

In [10]:
def encode(x):
    return [stoi[i] for i in x]

def decode(x):
    return ''.join([itos[i] for i in x])

In [11]:
print(encode("hellooo tghererer!!"))

[46, 43, 50, 50, 53, 53, 53, 1, 58, 45, 46, 43, 56, 43, 56, 43, 56, 2, 2]


In [12]:
print(decode([46, 43, 50, 50, 53, 53, 53, 1, 58, 45, 46, 43, 56, 43, 56, 43, 56, 2, 2]))

hellooo tghererer!!


In [13]:
data = torch.tensor(encode(text), dtype = torch.long)
data.dtype, data.shape

(torch.int64, torch.Size([1115394]))

In [14]:
t = int(0.9 * len(data))
train = data[:t]
val = data[t:]

In [16]:
block_size = 8
x = train[:block_size]
x

tensor([18, 47, 56, 57, 58,  1, 15, 47])

In [17]:
train[1:block_size]

tensor([47, 56, 57, 58,  1, 15, 47])

In [18]:
x = train[:block_size]
y = train[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    tar = y[i]
    print(f'context = {context} -> target = {tar}')

context = tensor([18]) -> target = 47
context = tensor([18, 47]) -> target = 56
context = tensor([18, 47, 56]) -> target = 57
context = tensor([18, 47, 56, 57]) -> target = 58
context = tensor([18, 47, 56, 57, 58]) -> target = 1
context = tensor([18, 47, 56, 57, 58,  1]) -> target = 15
context = tensor([18, 47, 56, 57, 58,  1, 15]) -> target = 47
context = tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> target = 58


In [19]:
x = train[:block_size]
y = train[1:block_size]
print(x,y)

tensor([18, 47, 56, 57, 58,  1, 15, 47]) tensor([47, 56, 57, 58,  1, 15, 47])


In [20]:
x[:1+1]

tensor([18, 47])

In [21]:
y[1]

tensor(56)

In [22]:
block_size = 8
batch_size = 4

def get_batch(value):
    data = train if value == 'train' else val
    ix = torch.randint(len(data)- block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    return x,y

In [23]:
xb, yb = get_batch("train")
print(xb.shape)
print(xb)

torch.Size([4, 8])
tensor([[59, 58, 46, 11,  0, 18, 53, 56],
        [44, 53, 56,  1, 51, 63,  1, 40],
        [ 1, 51, 63,  1, 57, 53, 59, 50],
        [39, 61,  1, 58, 46, 43,  1, 50]])


In [376]:
n_embed = 32
block_size = 8
batch_size = 4
num_head = 2
head_size = 16
dropout = 0.001

In [378]:
class Bigram(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_embed)
        self.lh1 = nn.Linear(n_embed, vocab_size)
        self.pos_embed = nn.Embedding(block_size, n_embed)
        self.head1 = Head(n_embed)
        self.mha = MultiHeadAttn(num_head, head_size) #this should return a b,t,n_embed 
        #self.layer = Block(num_head, head_size, n_embed)
        self.blocks = nn.Sequential(
            Block(num_head, head_size, n_embed),
            Block(num_head, head_size, n_embed),
            Block(num_head, head_size, n_embed),
            nn.LayerNorm(n_embed)
        )

    def forward(self, idx, targets=None): #idx is (B, T) if [[0,31]] shape is [1,2][b,t] 
        B, T = idx.shape
        tok_emb = self.embed(idx) # (B,T,C) after embed normally is (B,T,n_emebd)
        pos_emb = self.pos_embed(torch.arange(T)) #just plucks out till T row from pos_embed (T, n_embed) broadcastbale
        inp = tok_emb + pos_emb #b,t,n_embed goes in
        #h1 = self.head1(inp)
        #sa = self.mha(inp) # we send a b,t,n_embed that becomes b,t,head_sizwes and concat
        block = self.blocks(inp)
        logits = self.lh1(block)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_tokens):
        for _ in range(max_tokens):
            idx_cond = idx[:, -block_size:]
            logit, loss = self(idx_cond)
            logit = logit[:, -1, :] # take all batches wihich means nothing here take the last T which is what matters and all of its dimensions
            probs = F.softmax(logit, dim = -1)
            samp_idx = torch.multinomial(probs, num_samples = 1)
            idx = torch.cat((idx, samp_idx), dim=1)
            
        return idx

In [379]:
model= Bigram(vocab_size)

In [380]:
logits, loss= model(xb, yb) # B, T batch - 4 and T the 8 tokens in context

In [381]:
logits.shape

torch.Size([32, 65])

In [382]:
print(decode((model.generate(torch.zeros((1, 1), dtype=torch.long), 200))[0].tolist()))


lBmadMQ-vf EkFoCVZDyGU-mV-wUkb$aQLVVqwO!bDx JnlSmJwM$3LbNwPh33BEE$rnRT VpowO?XIf,xUpk-PmJkKGWZvEV IYkyxpSZ?Nfot-bDCb.LYWINDLhTI,rPW
&pBfMFfm3AxHo.FLY!JsbA!bFgINmZdzOrLZk-NR?uv$ZC:kAkGNnEc3zd BZPqTAMGv


In [383]:
optimizer = optim.AdamW(model.parameters(), lr = 1e-3)

In [384]:
for i in range(20000):
    if i%1000 == 0:
        print(loss.item())
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    
    #B,T,C = logits.shape
    #logits = logits.view(B*T, C)
    #tar = yb.view(B*T)
    
    #loss = F.cross_entropy(logits, tar)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(loss.item())

4.564462661743164
2.1535696983337402
2.033801317214966
2.5987133979797363
2.4330546855926514
2.8986949920654297
1.9398599863052368
1.957396388053894
1.623093843460083
2.2970430850982666
2.3231024742126465
2.144319534301758
2.032411575317383
1.7078081369400024
2.1581971645355225
1.7524641752243042
1.905299186706543
2.356969118118286
2.179428815841675
1.977351188659668
1.8682583570480347


In [387]:
print(decode((model.generate(torch.zeros((1, 1), dtype=torch.long), 500))[0].tolist()))


BRABELINIANUS:
For besterd thee booir sinch,
To do he man I:
For ofs is lefply corfore at whill withis this
Oriew bacge chear bay sin this E lackiles onter; be hard;
Thou brosh the be demard; ais from logein millone'd shals or labe, ho with is ooods: and word pritizends dame we hast lord's suse me he vestrend
Shus be riencel friman in'd the deids eevent.N LORIOSTENENRSARd nope fain.

Nad ar preconourys it that gud of in our hout you.

LOEINGLUJ:
Tha burgau, frevert a powers be quaft sentul, sow 


In [61]:
torch.arange(2)

tensor([0, 1])

In [65]:
""" Single head attention experinemt """

In [134]:
x = torch.randn(4,8,32)
B,T,C = x.shape #c is the embding dim of each token its embed + posembed

In [135]:
tril = torch.tril(torch.ones(8,8)) #this is the mask that follows

In [139]:
wei = torch.tril(torch.randn(8,8)) #cna have whatever

In [140]:
weight = wei.masked_fill(tril == 0, float('-inf')) #this masks to that whatever witht that upper trianlgle 0
weight

tensor([[-1.6875,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 1.5685, -1.6752,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 1.0570, -0.1659, -1.0960,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.6703, -1.6259,  2.1446,  0.3677,    -inf,    -inf,    -inf,    -inf],
        [ 0.0435,  1.6310, -0.6540,  0.9284, -0.0649,    -inf,    -inf,    -inf],
        [ 1.0496,  0.1023,  1.1699, -2.6681,  0.2407, -1.4562,    -inf,    -inf],
        [-0.4158,  0.3886,  0.1790, -0.4234, -2.4264, -1.3414, -2.2451,    -inf],
        [-0.3202,  0.2722, -0.0219, -1.6287,  1.3004,  0.3015,  1.9339,  1.1242]])

In [88]:
v = F.softmax(weight, dim = 1)
v.shape, v

(torch.Size([8, 8]),
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
         [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
         [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
         [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]]))

In [131]:
#cool now an attention head
head_size = 16

Wq = nn.Linear(C, head_size, bias = False)
Wk = nn.Linear(C, head_size, bias = False)
Wv = nn.Linear(C, head_size, bias = False)

query = Wq(x)  #B,T,C @ C, head ---> B, T, head so that 32 becam 16 for all tokens in each batch
key = Wk(x)  #same thing B,T,head so 4,8,16
#now calculate atteninon pattern q.k.T

attn_pattern = query @ key.transpose(-2, -1) * head_size**-0.5
attn_masked = attn_pattern.masked_fill(tril == 0, float('-inf'))
attn_masked_soft = F.softmax(attn_masked, dim = -1) #B,T,T
#attn_masked_soft[0]

value = Wv(x) #B,T,head now @ it with attn_masked_Sfot
value_mat = attn_masked_soft @ value 
value_mat.shape

torch.Size([4, 8, 16])

In [133]:
""" Experiment ends """

In [337]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        #gets in a T, n_embed make it T, head_size
        self.wq = nn.Linear(n_embed, head_size, bias = False)
        self.wk = nn.Linear(n_embed, head_size, bias = False)
        self.wv = nn.Linear(n_embed, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape # batch, T is full 8 in training and C is the n_embed
        
        query = self.wq(x) # b,t,c @ c,head --> b,t,head
        key = self.wk(x) # same thinf as above b,t,head so b,head,t @b,t,head transposeing
        
        attn_pattern = query @ key.transpose(-1,-2) * C**-0.5 #b,head,head
        attn_pattern_mask = attn_pattern.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        attn_softmax = F.softmax(attn_pattern_mask, dim = -1) #b,t,t

        value = self.wv(x) #b,t,head
        head_op = attn_softmax @ value 
        return head_op #b,t,head
        

In [344]:
class MultiHeadAttn(nn.Module):
    def __init__(self, num_head, head_size):
        super().__init__()
        self.head = nn.ModuleList([Head(head_size) for _ in range(num_head)])
        self.proj = nn.Linear(n_embed, n_embed) #to blend the glued up heads
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([heads(x) for heads in self.head], dim = -1) #we send b,t,n_embed we get concated b,t,head_size
        out = self.proj(out)
        out = self.drop(out)
        return out

In [345]:
class Feedforward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, n_embed * 4),
            nn.ReLU(),
            nn.Linear(n_embed *4, n_embed),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # this thing gets (B, T, n_embed)
        x = self.net(x) #return (B, T, n_embed)
        return x

In [346]:
class Block(nn.Module):
    def __init__(self, num_head, head_size, n_embed):
        super().__init__()

        self.mha = MultiHeadAttn(num_head, head_size) #gets in a T, n_embed
        #this calls Head whicht akes the n_embed and downproject it to head_size
        self.ffn = Feedforward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x =  x + self.mha(self.ln1(x)) # returns T, n_embed, also added residual additions , layernorm it and then send it
        x =  x + self.ffn(self.ln2(x)) # takes in a T,n_embed @ n_embed, n_embed --> T, n_embed
        # just make it T, vocab_size take last row and sammple
        return x

In [377]:
params = sum(p.numel() for p in model.parameters())
print(params)

49569


In [388]:
pwd

'C:\\Users\\Thilipan'